# Building An agent that has memory and access

1. source: https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv, https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=15
2. Note at the end we have function, loop_agent which is an agent without memory, I will also build an agant using langchain
2. Note: this is only for undersanding what the agent is:
3. framework of agent (steps that the agent does), note here we do not do the search by ourself like what did when building rag class:


    3.1. make a call to LLM (we pay here if we use openAI)
    
    3.2. LLM decides to invoke (perform) search('param') 
    
    3.3 We invoke the search, we have the results
    
    3.4. send the results back to the llm (another call), we pay again if we use open AI
    
    3.5. llm process the results
    
    3.6. llm decides to make another tool call
    
    3.7. send one more API request < --
    
    3.8. process, llm gives the answer (llm keep iterating)

4. The agent has 3 parts:
    
    4.1. Instruction, a developper prompt
    
    4.2. functions (tool) helper can call to carry out the task. For us that's only search.
    
    4.3. Memory, the message history. We append every prompt, every model output, and every tool 
    result. The agent reads this to know what it has already tried.

What is an agent, fundamentally?

An agent is just a loop:

User Question

      ↓

LLM thinks

      ↓

Need a tool?

      ↓

Use tool

      ↓

Observe result

      ↓

LLM thinks again

      ↓
      
Final answer

Plus:

Memory
Conversation History
State

## STEP 0, 1, 2, 3, & 4  for building an agent (has instructions, tools, and memory) using langchain

In [ ]:
# STEP 0, I need to check if the model behavious correctly (it has the ability to response correctly and uses tool, since not all the models use tool)
# from langchain_core.tools import tool
# from langchain_ollama import ChatOllama


# @tool
# def get_time() -> str:
#     """Get current time"""
#     return "10:00 AM"
# llm_to_test = ChatOllama(
#     model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
# )

# question = "what is the time"

# llm_to_test_with_tools = llm_to_test.bind_tools([get_time])

# response_from_llm_to_test = llm_to_test_with_tools.invoke(
#     question
# )

# print(response_from_llm_to_test.content) # the response is empty, because the model sees only the tool, I need to excute the tool and give the result back to llm
# print(response_from_llm_to_test.tool_calls)  # here it is not empty, so the model uses the tool
# print(response_from_llm_to_test.additional_kwargs)

In [ ]:
# NOTE the instruction is sooooooooooooooooo important
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from sqlitesearch import TextSearchIndex #sqliteindex 
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.messages import ToolMessage  # for tool call loop

###################################################################################
# Step 0, use sqlite elastic search to search a context from SQL DataBase
##################################################################################
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    # keyword_fiels=['course'],
    db_path='feq.db' # set the db name
)

#############################
# Step 1, set LLM
############################
llm = ChatOllama(
    model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
)

############################################################
# Step 2, set TOOL (functions, acess to YT, ....)
#############################################################
# NOTE::::::: The description of the fuction, is so important
@tool
def search(query: str) -> dict[str, str]: # when I use agent_tools.add, then the type will be added automatically and I do not need to set it.
    """
    Search the LLM Zoomcamp FAQ database.

    Use this tool whenever the user asks questions about:
    - joining the course
    - deadlines
    - registration
    - lectures
    - homework
    - certificates
    - course content
    """  
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

#########################################################
# Step 3, LLM with TOOL
########################################################
llm_with_tools = llm.bind_tools([search])

####################################################################################
# STEP 4 — Messages (like OpenAI) to set the instruction and the question, ....
####################################################################################
instructions = """You MUST use the search tool before answering.""".strip()

question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."

messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]


# the following has to be in a loop
"""
    # here I tell the LLM to use a tool
    response = llm_with_tools.invoke(
    messages
    )

    print('Check if the LLM has access to the TOOL Search, it will not answer anything here')
    print(response.content) # the response is empty, because the model sees only the tool, I need to excute the tool and give the result back to llm
    print(response.tool_calls)  # here it is not empty, so the model uses the tool
    print(response.additional_kwargs)
    ####################################################################################
    # 🔁 STEP 5 — TOOL CALL LOOP (THIS is your OpenAI loop)
    ####################################################################################
    # everytime, I need to append the message to keep the converstion on, if I did not append then 
    messages.append(response)

"""



# # Note this has to be in a loop, till I finish all the tools
# tool_call = response.tool_calls[0]
# if tool_call['name'] == 'search':
#     # now I need to exectue the serach, and then give it to LLM
#     # search is not a function, it is a tool_kit now, with parameters, search_tool = {
#     # "type": "function",
#     # "name": "search",
#     # "description": "Search the FAQ database for entries matching the given query.",
#     # "parameters": {
#     #     "type": "object",
#     #     "properties": {
#     #         "query": {
#     #             "type": "string",
#     #             "description": "Search query text to look up in the course FAQ."
#     #         }
#     #     },
#     #     "required": ["query"],
#     #     "additionalProperties": False}
   
#     result = search.invoke(tool_call["args"])
#     # I need to give the result to LLM, and I do not forget to append messages since we need to keep the conversation on
#     # append to message
#     messages.append(
#                 ToolMessage(
#                     content=str(result),
#                     tool_call_id=tool_call["id"]
#                 )
#             )



llm_with_tools = llm.bind_tools([search])

instructions = """You MUST use the search tool before answering.""".strip()

question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."

messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=question)
]


it = 1
MAX_ITERATIONS = 10  # this is to prevent repeating, becaue maybe the LLM will stuck, repeat search

while True:

    # to exit from the loop, when I finish all the tool
    has_tool_call = False
    print(f"\niteration #{it}...")

    # tell the llm to use tools
    response = llm_with_tools.invoke(messages)
    
    # now, I need the conversation to be on, so I need to update the state of the message
    messages.append(response)

    # STOP CONDITION (same as OpenAI)
    if not has_tool_calls:
        break

    # LLM has tools only, now I need to get answer from the tool and update the staus of the message
    # note not all LLM have tool, small model they do not have tool like gemma3:4b
    if response.tool_calls:
        for tool_call in response.tool_calls:
            # get the result from the tool
            if tool_call['name'] == 'search': # if the tool_call is name 
                tool_result = search.invoke(tool_call['args'])
                # append the result to messages
                messages.append(
                    ToolMessage(
                    content=str(tool_result), # give the content
                    tool_call_id=tool_call["id"] # the tool id
                )
                )
        # we need to iterate again and feed the LLM with the result from the tool, feed the LLM with the result
        has_tool_calls = True
    else:
        # 5. FINAL ANSWER (no tool calls)
        print("ASSISTANT:")
        print(response.content)

    it += 1
    if it > MAX_ITERATIONS:
        print("Maximum iterations reached.")
        break

    


In [ ]:
print(search)

### Build An Agent Manulally

In [ ]:

MAX_ITERATIONS = 10 
INSTRUCTIONS = """You are a FAQ assistant for the LLM Zoomcamp.

Always use the search tool before answering.

Answer ONLY using information returned by the search tool.

Keep answers concise and directly address the user's question.

Do not add unrelated information.

If the answer is not found in the search results, say:
"I don't know.""".strip()


# INSTRUCTIONS = """
# You're a course teaching assistant.
# You're given a question from a course student and your task is to answer it.

# If you want to look up information, use the search function. 
# Use as many keywords from the user question as possible when making first requests.

# Make multiple searches. First perform search, analyze the results 
# and then perform more searches. 

# At the end, ask if there are other areas that the user wants to explore.
# """.strip()
question = "Search the FAQ and tell me if I can join the LLM Zoomcamp."


llm = ChatOllama(
    model="qwen2.5:3b-instruct",#, # can not use since it does not have bind_tool "gemma3:4b" we can use llama3.1, .. gpt-5.4-mini
)
llm_with_tools = llm.bind_tools([search])



def agent_loop(
    question,
    instruction=INSTRUCTIONS,
):
    messages = [
        SystemMessage(content=instruction),
        HumanMessage(content=question)
    ]
    it = 0

    while it < MAX_ITERATIONS:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool call -> final answer
        if not response.tool_calls:
            return response.content

        # Execute tools
        for tool_call in response.tool_calls:
            if tool_call["name"] == "search":
                tool_result = search.invoke(tool_call["args"]) # search(**tool_call["args"])
                messages.append(
                    ToolMessage(
                        content=str(tool_result),
                        tool_call_id=tool_call["id"]
                    )
                )
        it += 1

    print('iteration =', it)
    return response.content

agent_loop(
    question,
    instruction=INSTRUCTIONS,
)

In [ ]:
agent_loop(INSTRUCTIONS, "How do I run Olama locally?")

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
agent_loop("what's queen gambit?")

### Build An Agent using langchain.agent
1. Tool-calling agent
2. RAG retrieval (RAG)
3. manual memory
4. langchain executer

In [3]:
from sqlitesearch import TextSearchIndex
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_ollama import ChatOllama
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    # keyword_fiels=['course'],
    db_path='feq.db' # set the db name
)

####  the search needs to return text
@tool
def search(query: str) -> dict[str, str]: # when I use agent_tools.add, then the type will be added automatically and I do not need to set it.
    """
    Search the LLM Zoomcamp FAQ database.

    Use this tool whenever the user asks questions about:
    - joining the course
    - deadlines
    - registration
    - lectures
    - homework
    - certificates
    - course content
    """  
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    search_results = index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()




question= "Can I still join the LLM Zoomcamp?"

INSTRUCTIONS = """You are a FAQ assistant for the LLM Zoomcamp.

Always use the search tool before answering.

Answer ONLY using information returned by the search tool.

Keep answers concise and directly address the user's question.

Do not add unrelated information.

If the answer is not found in the search results, say:
"I don't know.""".strip()


chat_history = []
#  1. prompt setting
prompt= ChatPromptTemplate.from_messages([
    ("system", INSTRUCTIONS),
    MessagesPlaceholder("chat_history"), # memory, long-term conversation memory (YOU control this)
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"), # internal working memory, It is the agent’s thinking workspace include: tool inputs, tool outputs, tool calls it decided to make, interal resoning steps,  (LangChain controls this), internal agent reasoning ( It stores:previous tool calls, tool outputs,reasoning steps,intermediate actions. The agent writes into this automatically.)
])

# 2. create llm
llm = ChatOllama(
    model="qwen2.5:3b-instruct",
)

# 3. create an agent
tools = [search]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

response = agent_executor.invoke({
    "input": question,
    "chat_history": chat_history,
})

# save history
chat_history.append(HumanMessage(content=question))
chat_history.append(AIMessage(content=response["output"]))

response



> Entering new AgentExecutor chain...

Invoking: `search` with `{'query': 'join LLM Zoomcamp'}`


General Course-Related Questions
Q: Where is the LLM Zoomcamp Telegram channel?
A: The Telegram channel is [https://t.me/llm_zoomcamp](https://t.me/llm_zoomcamp).

Use it for announcements. For technical discussion and questions, use the course Slack channel.

Capstone Project
Q: Where can I find previous LLM Zoomcamp projects?
A: You can browse previous LLM Zoomcamp project submissions here:

- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)
- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)

These pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.

General Course-Related Questions
Q: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?
A: Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

It c

{'input': 'Can I still join the LLM Zoomcamp?',
 'chat_history': [HumanMessage(content='Can I still join the LLM Zoomcamp?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Yes, you can still join the LLM Zoomcamp. However, if you wish to receive a certificate, you should submit your project while submission is ongoing. For tracking syllabus, deadlines, homework, and progress, use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'Yes, you can still join the LLM Zoomcamp. However, if you wish to receive a certificate, you should submit your project while submission is ongoing. For tracking syllabus, deadlines, homework, and progress, use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).'}

In [8]:
response['output']

'Yes, you can still join the LLM Zoomcamp. However, if you wish to receive a certificate, you should submit your project while submission is ongoing. For tracking syllabus, deadlines, homework, and progress, use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).'

# Wrap in a class

In [12]:
class FAQAgent:

    def __init__(self, agent_executor):

        self.agent_executor = agent_executor
        self.chat_history = []

    def ask(self, question: str) -> str:

        # 1. call agent
        response = self.agent_executor.invoke({
            "input": question,
            "chat_history": self.chat_history,
        })

        answer = response["output"]

        # 2. update memory
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=answer))

        # 3. optional: limit memory size
        self.chat_history = self.chat_history[-10:]

        return answer
    
agent = create_tool_calling_agent(
    llm=llm,
    tools=[search],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search],
    # verbose=True,
)

faq = FAQAgent(agent_executor)


print(faq.ask("Can I join the LLM Zoomcamp?"))

To join and participate in the LLM Zoomcamp, you can track the syllabus, deadlines, homework, and progress through the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/). If you want to receive a certificate, you need to submit your project while submissions are still accepted. Registration is for gauge interest before the start date.


In [13]:
print(faq.ask("Can I still  join the LLM Zoomcamp?"))

I don't know. Please check the syllabus or contact support for current information on joining the LLM Zoomcamp.


In [14]:
print(faq.ask("Do I need confirmation email?"))

I don't have specific details about whether a confirmation email is needed for joining the LLM Zoomcamp. You might want to check the course's registration page or contact support for this information.


In [2]:
import numpy as np
x = np.array([10,20,30,40,50,60])

np.argsort(x)[::-1]

array([5, 4, 3, 2, 1, 0])

In [3]:
np.argsort(-x)

array([5, 4, 3, 2, 1, 0])